# Day 9: Error Analysis, Feature Importance, and Model Review

**Project:** IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning

## Objectives
1. Read the Day 8 feature datasets and model-selection reports.
2. Recreate the same chronological 70% / 15% / 15% split so the diagnostics are comparable.
3. Refit the model selected by Day 8 validation MAE (RMSE tie-breaker) on the training partition only.
4. Analyse validation residuals and large errors; report test errors only as a final, non-selection diagnostic.
5. Calculate model-agnostic permutation importance on validation data and native importance when the selected estimator supports it.
6. Save reproducible CSV reports and a Markdown daily log.

**Leakage rules:** only numeric `prior_...` features are eligible; never select a model based on test metrics; do not treat importance as causal explanation. No metric is asserted until the notebook runs against the project's actual files.


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    XGBOOST_IMPORT_ERROR = ""
except Exception as exc:
    XGBOOST_AVAILABLE = False
    XGBOOST_IMPORT_ERROR = repr(exc)

warnings.filterwarnings("ignore", category=FutureWarning)
current_dir = Path.cwd().resolve()
candidates = [current_dir, *current_dir.parents]
def is_project_root(p):
    return (p / "data" / "processed" / "batting_next_match_features.csv").is_file() or ((p / "data" / "processed").is_dir() and (p / "notebooks").is_dir())
PROJECT_ROOT = next((p for p in candidates if is_project_root(p)), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\Git\Personal\ipl-player-performance")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DAY8_REPORT_DIR = PROJECT_ROOT / "reports" / "day8"
REPORT_DIR = PROJECT_ROOT / "reports" / "day9"
DOCS_DIR = PROJECT_ROOT / "docs"
for folder in [REPORT_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
BATTING_PATH = PROCESSED_DIR / "batting_next_match_features.csv"
BOWLING_PATH = PROCESSED_DIR / "bowling_next_match_features.csv"
SELECTION_PATH = DAY8_REPORT_DIR / "validation_model_selection.csv"
print("Project root:", PROJECT_ROOT)
for p in [BATTING_PATH, BOWLING_PATH, SELECTION_PATH]:
    print(p, "FOUND" if p.is_file() else "MISSING")
print("XGBoost available:", XGBOOST_AVAILABLE)
if not XGBOOST_AVAILABLE:
    print("XGBoost import note:", XGBOOST_IMPORT_ERROR)


Project root: D:\Git\Personal\ipl-player-performance
D:\Git\Personal\ipl-player-performance\data\processed\batting_next_match_features.csv FOUND
D:\Git\Personal\ipl-player-performance\data\processed\bowling_next_match_features.csv FOUND
D:\Git\Personal\ipl-player-performance\reports\day8\validation_model_selection.csv FOUND
XGBoost available: True


## 1. Load data and Day 8 decisions

This notebook intentionally requires the Day 8 selection report. It does not silently choose a new model if the report is missing.


In [2]:
def read_dataset(path, required):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run and verify Days 4, 7, and 8 first.")
    df = pd.read_csv(path)
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} missing columns {sorted(missing)}")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["next_match_date"] = pd.to_datetime(df["next_match_date"], errors="coerce")
    return df

batting = read_dataset(BATTING_PATH, {"player_name", "match_id", "date", "next_match_date", "next_match_runs"})
bowling = read_dataset(BOWLING_PATH, {"player_name", "match_id", "date", "next_match_date", "next_match_wickets", "next_match_runs_conceded"})
if not SELECTION_PATH.is_file():
    raise FileNotFoundError(f"Missing {SELECTION_PATH}. Run Day 8 successfully before Day 9.")
selection_df = pd.read_csv(SELECTION_PATH)
required_selection = {"dataset", "target", "selected_by_validation_mae"}
if not required_selection.issubset(selection_df.columns):
    raise ValueError(f"Day 8 selection report must contain {sorted(required_selection)}")
display(selection_df)
print("Batting shape:", batting.shape, "Bowling shape:", bowling.shape)


,dataset,target,selected_by_validation_mae,validation_mae,saved_pipeline
0,batting,next_match_runs,LinearRegression,15.864845,D:\Git\Personal\ipl-player-performance\models\...
1,bowling,next_match_wickets,LinearRegression,0.789086,D:\Git\Personal\ipl-player-performance\models\...
2,bowling,next_match_runs_conceded,XGBoost,8.490233,D:\Git\Personal\ipl-player-performance\models\...


Batting shape: (18030, 49) Bowling shape: (14123, 50)


## 2. Recreate approved features and chronological splits

Feature candidates are restricted to numeric `prior_...` columns. Review `reports/day7/feature_eligibility_review.csv` alongside this generated list before accepting model explanations.


In [3]:
TARGETS = {"batting": ["next_match_runs"], "bowling": ["next_match_wickets", "next_match_runs_conceded"]}
IDENTIFIERS = {"player_name", "player_id", "match_id", "next_match_id", "date", "next_match_date", "source_file"}
CURRENT_OUTCOMES = {"runs", "balls_faced", "fours", "sixes", "strike_rate", "batting_position", "dismissed", "wickets", "runs_conceded", "legal_balls", "overs", "economy", "maidens", "dots", "wides", "noballs"}

def get_prior_features(df):
    return [c for c in df.columns if c.startswith("prior_") and c not in IDENTIFIERS and c not in CURRENT_OUTCOMES and c not in sum(TARGETS.values(), []) and pd.api.types.is_numeric_dtype(df[c])]

def split_chronologically(df, target):
    work = df.loc[df[target].notna() & df["date"].notna()].copy()
    work = work.sort_values(["date", "match_id", "player_name"], kind="stable").reset_index(drop=True)
    if len(work) < 20:
        raise ValueError(f"{target}: only {len(work)} usable rows; cannot use the agreed split reliably.")
    train_end, val_end = int(len(work)*0.70), int(len(work)*0.85)
    if train_end < 2 or val_end <= train_end or val_end >= len(work):
        raise ValueError(f"Cannot create non-empty splits for {target}.")
    return work, work.iloc[:train_end].copy(), work.iloc[train_end:val_end].copy(), work.iloc[val_end:].copy()

def make_model(model_name):
    if model_name == "DummyMedian":
        estimator = DummyRegressor(strategy="median")
    elif model_name == "LinearRegression":
        estimator = LinearRegression()
    elif model_name == "RandomForest":
        estimator = RandomForestRegressor(n_estimators=300, max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=-1)
    elif model_name == "XGBoost" and XGBOOST_AVAILABLE:
        estimator = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, objective="reg:squarederror", random_state=42, n_jobs=-1)
    else:
        raise ValueError(f"Selected model '{model_name}' is not available in this environment.")
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])

def metrics(y, pred):
    return {"MAE": float(mean_absolute_error(y, pred)), "RMSE": float(np.sqrt(mean_squared_error(y, pred))), "R2": float(r2_score(y, pred)) if len(y) > 1 else np.nan}


## 3. Validation residual analysis and feature importance

Residual is `actual - predicted`. Positive residual means the model under-predicted; negative means it over-predicted. The validation set is used for error analysis and permutation importance. The test set is evaluated only as a final diagnostic and is not used for model selection or feature ranking.


In [4]:
error_rows, importance_rows, native_importance_rows, split_rows, status_rows = [], [], [], [], []
TASKS = [(batting, "batting", "next_match_runs"), (bowling, "bowling", "next_match_wickets"), (bowling, "bowling", "next_match_runs_conceded")]

for df, dataset_name, target in TASKS:
    chosen_rows = selection_df[(selection_df["dataset"].astype(str) == dataset_name) & (selection_df["target"].astype(str) == target)]
    if chosen_rows.empty:
        status_rows.append({"dataset": dataset_name, "target": target, "status": "SKIPPED", "reason": "No matching Day 8 model-selection row."})
        continue
    selected_model_name = str(chosen_rows.iloc[0]["selected_by_validation_mae"])
    work, train, val, test = split_chronologically(df, target)
    features = get_prior_features(df)
    features = [c for c in features if train[c].notna().any()]
    if not features:
        status_rows.append({"dataset": dataset_name, "target": target, "status": "SKIPPED", "reason": "No numeric prior-history features available in training."})
        continue
    X_train = train[features].apply(pd.to_numeric, errors="coerce")
    X_val = val[features].apply(pd.to_numeric, errors="coerce")
    X_test = test[features].apply(pd.to_numeric, errors="coerce")
    y_train = pd.to_numeric(train[target], errors="coerce")
    y_val = pd.to_numeric(val[target], errors="coerce")
    y_test = pd.to_numeric(test[target], errors="coerce")
    model = make_model(selected_model_name)
    model.fit(X_train, y_train)
    split_rows.append({"dataset": dataset_name, "target": target, "selected_model": selected_model_name, "rows": len(work), "train_rows": len(train), "validation_rows": len(val), "test_rows": len(test), "train_start": str(train.date.min().date()), "train_end": str(train.date.max().date()), "validation_start": str(val.date.min().date()), "validation_end": str(val.date.max().date()), "test_start": str(test.date.min().date()), "test_end": str(test.date.max().date()), "feature_count": len(features)})
    for split_name, part, X_part, y_part in [("validation", val, X_val, y_val), ("test_final_diagnostic", test, X_test, y_test)]:
        pred = model.predict(X_part)
        for idx, (_, row) in enumerate(part.iterrows()):
            actual = float(y_part.iloc[idx]); predicted = float(pred[idx]); residual = actual - predicted
            record = {"dataset": dataset_name, "target": target, "split": split_name, "player_name": row.get("player_name", ""), "match_id": row.get("match_id", ""), "date": row["date"], "next_match_date": row["next_match_date"], "actual": actual, "predicted": predicted, "residual_actual_minus_predicted": residual, "absolute_error": abs(residual)}
            error_rows.append(record)
    # Validation permutation importance: shuffle one feature at a time and measure increase in MAE.
    perm = permutation_importance(model, X_val, y_val, scoring="neg_mean_absolute_error", n_repeats=10, random_state=42, n_jobs=-1)
    for feature, mean_imp, std_imp in zip(features, perm.importances_mean, perm.importances_std):
        importance_rows.append({"dataset": dataset_name, "target": target, "selected_model": selected_model_name, "feature": feature, "permutation_importance_mae_increase": float(mean_imp), "importance_std": float(std_imp), "evaluation_split": "validation"})
    fitted_estimator = model.named_steps["model"]
    if hasattr(fitted_estimator, "feature_importances_"):
        for feature, value in zip(features, fitted_estimator.feature_importances_):
            native_importance_rows.append({"dataset": dataset_name, "target": target, "selected_model": selected_model_name, "feature": feature, "native_importance": float(value), "method": "tree_feature_importances_"})
    status_rows.append({"dataset": dataset_name, "target": target, "status": "COMPLETED", "reason": f"Selected by Day 8 validation MAE: {selected_model_name}"})

errors_df = pd.DataFrame(error_rows)
importance_df = pd.DataFrame(importance_rows)
native_importance_df = pd.DataFrame(native_importance_rows)
splits_df = pd.DataFrame(split_rows)
status_df = pd.DataFrame(status_rows)
for name, frame in [("status", status_df), ("splits", splits_df)]:
    print(name.upper()); display(frame)
print("Validation error sample:"); display(errors_df[errors_df["split"] == "validation"].head(10) if not errors_df.empty else errors_df)
print("Top permutation importance:"); display(importance_df.sort_values(["dataset", "target", "permutation_importance_mae_increase"], ascending=[True, True, False]).groupby(["dataset", "target"], as_index=False, group_keys=False).head(10) if not importance_df.empty else importance_df)


STATUS


,dataset,target,status,reason
0,batting,next_match_runs,COMPLETED,Selected by Day 8 validation MAE: LinearRegres...
1,bowling,next_match_wickets,COMPLETED,Selected by Day 8 validation MAE: LinearRegres...
2,bowling,next_match_runs_conceded,COMPLETED,Selected by Day 8 validation MAE: XGBoost


SPLITS


,dataset,target,selected_model,rows,train_rows,validation_rows,test_rows,train_start,train_end,validation_start,validation_end,test_start,test_end,feature_count
0,batting,next_match_runs,LinearRegression,18030,12621,2704,2705,2008-04-18,2021-10-08,2021-10-08,2024-04-13,2024-04-13,2026-05-29,15
1,bowling,next_match_wickets,LinearRegression,14123,9886,2118,2119,2008-04-18,2021-10-01,2021-10-01,2024-04-08,2024-04-08,2026-05-29,15
2,bowling,next_match_runs_conceded,XGBoost,14123,9886,2118,2119,2008-04-18,2021-10-01,2021-10-01,2024-04-08,2024-04-08,2026-05-29,15


Validation error sample:


,dataset,target,split,player_name,match_id,date,next_match_date,actual,predicted,residual_actual_minus_predicted,absolute_error
0,batting,next_match_runs,validation,V Kohli,1254101,2021-10-08,2021-10-11,39.0,28.201084,10.798916,10.798916
1,batting,next_match_runs,validation,AR Patel,1254114,2021-10-10,2021-10-13,4.0,14.638680,-10.638680,10.638680
2,batting,next_match_runs,validation,AT Rayudu,1254114,2021-10-10,2022-03-26,15.0,22.692410,-7.692410,7.692410
3,batting,next_match_runs,validation,F du Plessis,1254114,2021-10-10,2021-10-15,86.0,29.802978,56.197022,56.197022
4,batting,next_match_runs,validation,MM Ali,1254114,2021-10-10,2021-10-15,37.0,20.186125,16.813875,16.813875
5,batting,next_match_runs,validation,MS Dhoni,1254114,2021-10-10,2022-03-26,50.0,21.443007,28.556993,28.556993
6,batting,next_match_runs,validation,PP Shaw,1254114,2021-10-10,2021-10-13,18.0,21.472955,-3.472955,3.472955
7,batting,next_match_runs,validation,RD Gaikwad,1254114,2021-10-10,2021-10-15,32.0,32.160176,-0.160176,0.160176
8,batting,next_match_runs,validation,RR Pant,1254114,2021-10-10,2021-10-13,6.0,24.927250,-18.927250,18.927250
9,batting,next_match_runs,validation,RV Uthappa,1254114,2021-10-10,2021-10-15,31.0,21.153320,9.846680,9.846680


Top permutation importance:


,dataset,target,selected_model,feature,permutation_importance_mae_increase,importance_std,evaluation_split
3,batting,next_match_runs,LinearRegression,prior_career_batting_balls_faced,1.605897,0.074373,validation
1,batting,next_match_runs,LinearRegression,prior_last5_mean_batting_runs,0.817615,0.058521,validation
14,batting,next_match_runs,LinearRegression,prior_last5_count_batting_strike_rate,0.763914,0.065864,validation
9,batting,next_match_runs,LinearRegression,prior_career_batting_sixes,0.233450,0.029606,validation
0,batting,next_match_runs,LinearRegression,prior_career_batting_runs,0.189351,0.045004,validation
6,batting,next_match_runs,LinearRegression,prior_career_batting_fours,0.106334,0.014778,validation
7,batting,next_match_runs,LinearRegression,prior_last5_mean_batting_fours,0.048013,0.010873,validation
12,batting,next_match_runs,LinearRegression,prior_career_batting_strike_rate,0.026844,0.009481,validation
13,batting,next_match_runs,LinearRegression,prior_last5_mean_batting_strike_rate,0.007157,0.023348,validation
5,batting,next_match_runs,LinearRegression,prior_last5_count_batting_balls_faced,-0.014565,0.013113,validation


## 4. Aggregate errors and export reports

Review mean absolute error, RMSE, bias, and large-error records by target. Large errors are diagnostic examples, not proof that the model is wrong; verify source data and target alignment before drawing conclusions.


In [5]:
if not errors_df.empty:
    aggregate_rows = []
    for (dataset_name, target, split_name), group in errors_df.groupby(["dataset", "target", "split"]):
        aggregate_rows.append({"dataset": dataset_name, "target": target, "split": split_name, "rows": len(group), **metrics(group["actual"], group["predicted"]), "mean_residual_actual_minus_predicted": float(group["residual_actual_minus_predicted"].mean()), "median_absolute_error": float(group["absolute_error"].median()), "p90_absolute_error": float(group["absolute_error"].quantile(0.90)), "max_absolute_error": float(group["absolute_error"].max())})
    error_summary_df = pd.DataFrame(aggregate_rows)
    large_errors_df = (errors_df[errors_df["split"] == "validation"].sort_values(["dataset", "target", "absolute_error"], ascending=[True, True, False]).groupby(["dataset", "target"], group_keys=False).head(20).copy())
else:
    error_summary_df = pd.DataFrame()
    large_errors_df = pd.DataFrame()

outputs = {
    "error_predictions.csv": errors_df,
    "error_summary.csv": error_summary_df,
    "largest_validation_errors.csv": large_errors_df,
    "permutation_feature_importance.csv": importance_df,
    "native_tree_feature_importance.csv": native_importance_df,
    "split_summary.csv": splits_df,
    "task_status.csv": status_df,
}
for filename, frame in outputs.items():
    frame.to_csv(REPORT_DIR / filename, index=False)
    print("Saved:", REPORT_DIR / filename, "rows:", len(frame))

display(error_summary_df)
display(large_errors_df.head(20) if not large_errors_df.empty else large_errors_df)


Saved: D:\Git\Personal\ipl-player-performance\reports\day9\error_predictions.csv rows: 13883
Saved: D:\Git\Personal\ipl-player-performance\reports\day9\error_summary.csv rows: 6
Saved: D:\Git\Personal\ipl-player-performance\reports\day9\largest_validation_errors.csv rows: 60
Saved: D:\Git\Personal\ipl-player-performance\reports\day9\permutation_feature_importance.csv rows: 45
Saved: D:\Git\Personal\ipl-player-performance\reports\day9\native_tree_feature_importance.csv rows: 15
Saved: D:\Git\Personal\ipl-player-performance\reports\day9\split_summary.csv rows: 3
Saved: D:\Git\Personal\ipl-player-performance\reports\day9\task_status.csv rows: 3


,dataset,target,split,rows,MAE,RMSE,R2,mean_residual_actual_minus_predicted,median_absolute_error,p90_absolute_error,max_absolute_error
0,batting,next_match_runs,test_final_diagnostic,2705,16.835979,22.663426,0.101603,2.821742,13.220589,34.663988,121.722479
1,batting,next_match_runs,validation,2704,15.864845,21.231884,0.105058,1.268018,12.366859,31.896621,114.401996
2,bowling,next_match_runs_conceded,test_final_diagnostic,2119,9.767157,12.360691,-0.002108,4.187010,8.150419,19.876905,49.359316
3,bowling,next_match_runs_conceded,validation,2118,8.490233,10.764133,0.115517,1.870741,7.039113,17.559795,45.047338
4,bowling,next_match_wickets,test_final_diagnostic,2119,0.767192,0.990279,0.024471,0.054471,0.886932,1.283623,4.186592
5,bowling,next_match_wickets,validation,2118,0.789086,1.036961,0.027026,0.047310,0.884755,1.289294,4.255998


,dataset,target,split,player_name,match_id,date,next_match_date,actual,predicted,residual_actual_minus_predicted,absolute_error
1023,batting,next_match_runs,validation,Q de Kock,1304109,2022-05-15,2022-05-18,140.0,25.598004,114.401996,114.401996
1781,batting,next_match_runs,validation,YBK Jaiswal,1359511,2023-04-27,2023-04-30,124.0,26.561859,97.438141,97.438141
2268,batting,next_match_runs,validation,Shubman Gill,1370350,2023-05-23,2023-05-26,129.0,32.790777,96.209223,96.209223
1394,batting,next_match_runs,validation,HC Brook,1359488,2023-04-09,2023-04-14,100.0,12.653661,87.346339,87.346339
978,batting,next_match_runs,validation,RM Patidar,1304106,2022-05-13,2022-05-25,112.0,25.085391,86.914609,86.914609
505,batting,next_match_runs,validation,JC Buttler,1304076,2022-04-18,2022-04-22,116.0,29.201478,86.798522,86.798522
2006,batting,next_match_runs,validation,P Simran Singh,1359527,2023-05-08,2023-05-13,103.0,19.272301,83.727699,83.727699
2138,batting,next_match_runs,validation,H Klaasen,1359536,2023-05-15,2023-05-18,104.0,20.275137,83.724863,83.724863
2680,batting,next_match_runs,validation,RG Sharma,1426263,2024-04-11,2024-04-14,105.0,24.963161,80.036839,80.036839
2151,batting,next_match_runs,validation,C Green,1359537,2023-05-16,2023-05-21,100.0,20.031601,79.968399,79.968399


## 5. Interpretation checklist

- [ ] Verify the Day 8 selected model was carried forward for each target.
- [ ] Compare validation error summary with the final test diagnostic; do not use test scores to change the selected model.
- [ ] Review mean residual: a positive value indicates under-prediction on average; negative indicates over-prediction.
- [ ] Inspect the largest validation errors against source records and next-match target alignment.
- [ ] Check whether the highest-ranked features make cricket/domain sense and were genuinely known before the target match.
- [ ] Treat permutation importance as model-dependent association, not causality. Negative values can occur due to sampling variability.
- [ ] Review row-based split boundaries: rows from the same match may straddle splits. This remains a limitation to address before final reporting.
- [ ] Do not deploy or present a pipeline as production-ready solely because this notebook completes.


In [6]:
def text_table(frame):
    return frame.to_string(index=False) if frame is not None and not frame.empty else "No rows produced."

doc = f"""# Day 9: Error Analysis, Feature Importance, and Model Review

## Status
Notebook execution status is represented by the task status report below. No metrics are hard-coded into this document.

## Task status
```text
{text_table(status_df)}
```

## Split summary
```text
{text_table(splits_df)}
```

## Error summary
```text
{text_table(error_summary_df)}
```

## Top validation permutation importance
```text
{text_table(importance_df.sort_values('permutation_importance_mae_increase', ascending=False).groupby(['dataset','target'], as_index=False, group_keys=False).head(10) if not importance_df.empty else importance_df)}
```

## Files produced
- `reports/day9/error_predictions.csv`
- `reports/day9/error_summary.csv`
- `reports/day9/largest_validation_errors.csv`
- `reports/day9/permutation_feature_importance.csv`
- `reports/day9/native_tree_feature_importance.csv`
- `reports/day9/split_summary.csv`
- `reports/day9/task_status.csv`
- `docs/day9_error_analysis_feature_importance.md`

## Methodology
- The Day 8 validation-selected model is reused for each target.
- Model is refit on the training partition only; validation supports diagnostics and permutation importance.
- Test partition metrics are a final diagnostic only and must not drive selection.
- Numeric `prior_...` columns are the only automatic predictor candidates.
- Metrics: MAE, RMSE, R², residual bias, median/p90/max absolute error.
- Feature importance is descriptive, model-dependent, and not causal.

## Limitations and decisions
- The chronological split is row-based (70/15/15); records from one match can cross boundaries.
- Permutation importance can be noisy for correlated features or small validation samples.
- A high-error row must be investigated against the source, target alignment, and player-match chronology before data is changed.
- No model should be promoted to production based on a single diagnostic notebook.

## Next starting point
Day 10: consolidate model evaluation and define the final training/evaluation protocol, including a safer match-date/group-aware split if the data supports it. Do not start Day 10 until Day 9 outputs and documentation are reviewed.
"""
DOC_PATH = DOCS_DIR / "day9_error_analysis_feature_importance.md"
DOC_PATH.write_text(doc, encoding="utf-8")
print("Saved documentation:", DOC_PATH)


Saved documentation: D:\Git\Personal\ipl-player-performance\docs\day9_error_analysis_feature_importance.md
